(sec-iter-top)=

<div align="center">
    <h1>Iterative Development</h1>
    <a href="https://github.com/bernalde">David E. Bernal Neira</a>
    <br>
    <i>Davidson School of Chemical Engineering, Purdue University</i>
    <br>
    <a href="https://colab.research.google.com/github/SECQUOIA/PU_CHE597_DSinChemE/blob/main/6-Iterative_Code_Development/Iterative_Development.ipynb" target="_parent">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
    <a href="https://secquoia.github.io/">
        <img src="https://img.shields.io/badge/🌲⚛️🌐-SECQUOIA-blue" alt="SECQUOIA"/>
    </a>
</div>


(sec-iter-dev)=
# Iterative Development
## Table of Contents
- [Iterative Development](#sec-iter-dev)


In [ ]:
# If using this on Google colab, we need to install the packages
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False


<b>If you are using Google Colab you should save this notebook and any associated text files to their own folder on your Google Drive. Then you will need to adapt the following commands so that the notebook runs from the location of that folder.</b>


In [ ]:
# If you want to use Google Drive to save/load files, set this to True
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    # Colab command to navigate to the folder holding the homework,
    # CHANGE FOR YOUR SPECIFIC FOLDER LOCATION IN GOOGLE DRIVE
    # Note: if there are spaces in the path, you need to precede them with a backslash '\'
    %cd /content/drive/My\ Drive/CHE597/Lectures/iterative_development


# Iterative Development

When you are writing complex programs, or even just multi-step functions or loops, it is critical that you break things into digestible chunks. The term <b>iterative development</b> describes this necessity to iterate on a piece of code, until it does everything you want. 

On HW 2, I asked you to write two functions that required several steps each: writing a window average function, and writing a parser for a file with chained data for multiple reactors. In each of these problems you had to write functions that did several things at once. But starting from scratch, it is very difficult to write something that does everything on the first try. It is much easier to write pieces of the functions, test as you go, and incrementally build up the coding solution. In this notebook, I will show you an example of how iterative development works. 

Here is the `multi_reactor.txt` parsing problem from the homework:

Note: See Python file I/O: https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files



In [3]:
# The file mult_reactors.txt has time vs impurity data for several reactors
# Inspect the file.
# Write your own parser that reads each reactor's data into a separate array
# and assigns these arrays to a dictionary with pointer reactors.
# NOTE: np.genfromtxt() and np.loadtxt() will not work for this task
import numpy as np

import os
if not os.path.exists('mult_reactors.txt'):
    !wget -q https://raw.githubusercontent.com/SECQUOIA/PU_CHE597_DSinChemE/main/6-Iterative_Code_Development/mult_reactors.txt -O mult_reactors.txt


This problem asks us to 1) open a file for reading and iteration, 2) distinguish between different reactors, and 3) save the data in a specific format (arrays within a dictionary).

Taking an iterative approach we will break this down into elementary tasks. Specifically, we will start with reading data, then just try and parse the first reactor's data, before finally trying to parse all of the reactors at once. Eventually we will arrive at a relatively sophisticated solution, but each of the individual steps are quite straightforward.  

 Let's first write a solution for opening the file and iterating over its contents:


In [4]:
with open("mult_reactors.txt",'r') as f:
  for count, lines in enumerate(f):
    print(lines)

    # For diagnostic purposes
    if count == 10:
      break    


########################################

# Reactor 1 Data

########################################

 Time(min)     Impurity(mg/L)

 0.000000     73.826827  

 1.000000     74.152555  

 2.000000     73.246178  

 3.000000     73.748624  

 4.000000     73.486195  

 5.000000     74.382252  

 6.000000     74.076703  



Mini-exercise: Print the first non-empty line in the file.


In [5]:
# Mini-exercise
# Print the first non-empty line.
line = None  # TODO: set this to the first non-empty line
if line is None:
    print('TODO: find the first non-empty line')
else:
    print(line.strip())
# Expected output: a header line like "Time(min)" or a data line.


TODO: find the first non-empty line


The code above is responsible for opening the file and reading it line by line. We've also used the `count/enumerate` construction to test it on the first few lines. We won't keep this around in the final solution, but putting in this kind of diagnostic scaffolding is important while we're developing our solution. Now, let's add code so that we only get the parts of the file that are data (as opposed to all of the strings and headers):


In [6]:
with open("mult_reactors.txt",'r') as f:
  for count, lines in enumerate(f):
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Get the numbers
    try:
      print("{} {}".format(float(fields[0]),float(fields[1]))) # only numbers will survive this try
    except:
      pass

    # For diagnostic purposes
    if count == 10:
      break 


0.0 73.826827
1.0 74.152555
2.0 73.246178
3.0 73.748624
4.0 73.486195
5.0 74.382252
6.0 74.076703


Mini-exercise: Parse the first data line into two floats.


In [7]:
# Mini-exercise
# Parse the first data line into two floats.
vals = None  # TODO: set to a list of two floats
if vals is None:
    print('TODO: parse a data line')
else:
    print(vals)
# Expected output: two numeric values.


TODO: parse a data line


So far so good, we are (1) reading the file and (2) only grabbing the data. Now, we want to save the data, not just print it. Since we are reading in line by line, we will need to use an object that is good for appending values, like a list:


In [8]:
data = []
with open("mult_reactors.txt",'r') as f:
  for count, lines in enumerate(f):
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Get the numbers
    try:
      data += [float(fields[0]),float(fields[1])]
    except:
      pass

    # For diagnostic purposes
    if count == 10:
      break 
print(data)


[0.0, 73.826827, 1.0, 74.152555, 2.0, 73.246178, 3.0, 73.748624, 4.0, 73.486195, 5.0, 74.382252, 6.0, 74.076703]


Now we are saving the data, but we can see a problem: it isn't keeping the rows separate in the list, it is just combining all of the rows. To keep the rows separate, it is better to save the data as a list of lists, or list of tuples:


In [9]:
data = []
with open("mult_reactors.txt",'r') as f:
  for count, lines in enumerate(f):
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Get the numbers
    try:
      data += [[float(fields[0]),float(fields[1])]]
    except:
      pass

    # For diagnostic purposes
    if count == 10:
      break 
print(data)


[[0.0, 73.826827], [1.0, 74.152555], [2.0, 73.246178], [3.0, 73.748624], [4.0, 73.486195], [5.0, 74.382252], [6.0, 74.076703]]


Now, the data from row `0` is in the list at index `0`, and so on. 

Next we know that there are several independent reactors. Each separated by a header. Let's add the code necessary to <b>only get the data from the first reactor</b>. This is a good iterative strategy. After we get the first reactor working, we can deal with the other reactors.

To only get the first reactor's data we will add a "counter" or "iteration" variable `N_reactor`, to keep track of which reactor we are parsing, and we will need to add an `if` statment of some kind to figure out when the reactor's data ends. We will also update our `break` statement to exit the loop after we think that we have the first reactor's data:


In [10]:
data = [] # will temporarily hold each reactor's data
N_reactor = -1 # keeps track of which reactor we are parsing
with open("mult_reactors.txt",'r') as f:
  for lines in f:
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Skip empty lines
    if len(fields) == 0:
      continue

    # Identify start of new reactor data 
    if fields[0] == "Time(min)":
      N_reactor += 1

    # Get the numbers
    try:
      data += [[float(fields[0]),float(fields[1])]]
    except:
      pass

    # For diagnostic purposes
    if N_reactor == 1:
      break
print(data)


[[0.0, 73.826827], [1.0, 74.152555], [2.0, 73.246178], [3.0, 73.748624], [4.0, 73.486195], [5.0, 74.382252], [6.0, 74.076703], [7.0, 74.444489], [8.0, 75.093759], [9.0, 75.066265], [10.0, 76.034617], [11.0, 76.242245], [12.0, 76.761494], [13.0, 77.130528], [14.0, 77.99536], [15.0, 78.894931], [16.0, 79.875153], [17.0, 79.127464], [18.0, 80.080665], [19.0, 79.538877], [20.0, 78.91099], [21.0, 78.929412], [22.0, 78.990408], [23.0, 78.555686], [24.0, 78.176009], [25.0, 77.795778], [26.0, 77.328556], [27.0, 77.515418], [28.0, 76.949687], [29.0, 76.754453], [30.0, 76.331575], [31.0, 75.679257], [32.0, 75.130684], [33.0, 76.107967], [34.0, 75.976069], [35.0, 76.632448], [36.0, 76.777474], [37.0, 77.081274], [38.0, 77.540751], [39.0, 77.323318], [40.0, 77.109043], [41.0, 77.828145], [42.0, 78.448184], [43.0, 79.121272], [44.0, 78.497248], [45.0, 78.216442], [46.0, 78.180409], [47.0, 79.112555], [48.0, 79.239181], [49.0, 79.894191], [50.0, 79.046406], [51.0, 79.152682], [52.0, 79.248563], [53.

We've added a way of finding the end of the reactor's data by using the header entry `"Time(min)"`. Every time that the parser encounters such a line, a new reactor is about to be parsed. We've also updated our diagnostic `break` statement to break after the first reactor has been parsed (i.e., when `N_reactor == 1`) and we've removed the `enumerate()` function because it isn't needed any longer.

However, we've encountered a problem: `IndexError: list index out of range`. This means that we've tried to access an element `fields[0]` which is out of range for the list. Since `0` is the first element, this means that during this iteration the list was empty. Let's add a `print` statement to see what was happening right before the failure:


In [11]:
data = [] # will temporarily hold each reactor's data
N_reactor = -1 # keeps track of which reactor we are parsing
with open("mult_reactors.txt",'r') as f:
  for lines in f:
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # FIND OUT WHAT WAS HAPPENING BEFORE THE ERROR
    print(fields)

    # Identify start of new reactor data
    if fields[0] == "Time(min)":
      N_reactor += 1

    # Get the numbers
    try:
      data += [[float(fields[0]),float(fields[1])]]
    except:
      pass

    # For diagnostic purposes
    if N_reactor == 1:
      break
#print(data)


['########################################']
['#', 'Reactor', '1', 'Data']
['########################################']
['Time(min)', 'Impurity(mg/L)']
['0.000000', '73.826827']
['1.000000', '74.152555']
['2.000000', '73.246178']
['3.000000', '73.748624']
['4.000000', '73.486195']
['5.000000', '74.382252']
['6.000000', '74.076703']
['7.000000', '74.444489']
['8.000000', '75.093759']
['9.000000', '75.066265']
['10.000000', '76.034617']
['11.000000', '76.242245']
['12.000000', '76.761494']
['13.000000', '77.130528']
['14.000000', '77.995360']
['15.000000', '78.894931']
['16.000000', '79.875153']
['17.000000', '79.127464']
['18.000000', '80.080665']
['19.000000', '79.538877']
['20.000000', '78.910990']
['21.000000', '78.929412']
['22.000000', '78.990408']
['23.000000', '78.555686']
['24.000000', '78.176009']
['25.000000', '77.795778']
['26.000000', '77.328556']
['27.000000', '77.515418']
['28.000000', '76.949687']
['29.000000', '76.754453']
['30.000000', '76.331575']
['31.000000', '75.679

IndexError: list index out of range

The printout is long. This is because the parser worked for a long time before encountering an error. Looking at the last printed statement before the error, we see an empty list (`[]`). Inspecting the file, we would see that there is an empty line between the reactors. Now that we see it, we can easily deal with this by skipping empty lines. Let's try again:


In [12]:
data = [] # will temporarily hold each reactor's data
N_reactor = -1 # keeps track of which reactor we are parsing
with open("mult_reactors.txt",'r') as f:
  for lines in f:
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Skip empty
    if len(fields) == 0:
      continue

    # Identify start of new reactor data
    if fields[0] == "Time(min)":
      N_reactor += 1

    # Get the numbers
    try:
      data += [[float(fields[0]),float(fields[1])]]
    except:
      pass

    # For diagnostic purposes
    if N_reactor == 1:
      break
print("length of data: {}".format(len(data)))
print("data[0]: {}".format(data[0]))
print("data[-1]: {}".format(data[-1]))


length of data: 481
data[0]: [0.0, 73.826827]
data[-1]: [480.0, 83.268171]


Skipping empty lines has now solved our problem. The code executes without error through the first reactor, and we have successfully saved all of its data to `data`. 

Before parsing the next reactor, we need to put the first reactor's data somewhere. Specifically, from the problem we know what we should save it as an array within a dictionary. Let's add this functionality before trying to parse the rest of the reactors. 


In [13]:
data = [] # will temporarily hold each reactor's data
N_reactor = -1 # keeps track of which reactor we are parsing
reactors = {} # dictionary for holding each reactor's data as an array
with open("mult_reactors.txt",'r') as f:
  for lines in f:
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Skip empty
    if len(fields) == 0:
      continue

    # Identify start of new reactor data
    if fields[0] == "Time(min)":
      reactors[N_reactor] = np.array(data)
      N_reactor += 1

    # Get the numbers
    try:
      data += [[float(fields[0]),float(fields[1])]]
    except:
      pass

    # For diagnostic purposes
    if N_reactor == 1:
      break
print("reactors.keys(): {}".format(reactors.keys()))
print("lengths: {}".format([ len(reactors[i]) for i in reactors.keys() ]))


reactors.keys(): dict_keys([-1, 0])
lengths: [0, 481]


Mini-exercise: Count how many reactors are in the file.


In [14]:
# Mini-exercise
# Count the number of reactors by counting header lines.
count = None  # TODO: compute count
if count is None:
    print('TODO: count reactors')
else:
    print(count)
# Expected output: an integer (e.g., 10).


TODO: count reactors


To check our work we might have printed out the whole `reactors` dictionary and we would have seen an issue. To save space I've just printed out the `keys()` and lengths of each array in `reactors`. The issue is that we have two arrays saved, with keys `-1` and `0` respectively, and the first one is empty. What has happened? Take a minute and think about it before moving on. 

The issue is that the first time that we encounter `"Time(min)"` is at the top of the file, <i>before</i> we have parsed any reactor data. This first encounter with `"Time(min)"` needs to be skipped. An easy way to do this is to only save the `data` if it is non-empty: 


In [15]:
data = [] # will temporarily hold each reactor's data
N_reactor = -1 # keeps track of which reactor we are parsing
reactors = {} # dictionary for holding each reactor's data as an array
with open("mult_reactors.txt",'r') as f:
  for lines in f:
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Skip empty
    if len(fields) == 0:
      continue

    # Identify start of new reactor data
    if fields[0] == "Time(min)":

      # Save if non-empty
      if data:
        reactors[N_reactor] = np.array(data)
      N_reactor += 1

    # Get the numbers
    try:
      data += [[float(fields[0]),float(fields[1])]]
    except:
      pass

    # For diagnostic purposes
    if N_reactor == 1:
      break
print("reactors.keys(): {}".format(reactors.keys()))
print("lengths: {}".format([ len(reactors[i]) for i in reactors.keys() ]))


reactors.keys(): dict_keys([0])
lengths: [481]


With this change, we are now successfully parsing the first reactor's data and assigning it to the dictionary as an array. 

We've done almost all of the work. The code as written should work on the rest of the reactors, with two modifications. But first let's just test it to see what it does when we let it loose on the rest of the file:


In [16]:
data = [] # will temporarily hold each reactor's data
N_reactor = -1 # keeps track of which reactor we are parsing
reactors = {} # dictionary for holding each reactor's data as an array
with open("mult_reactors.txt",'r') as f:
  for lines in f:
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Skip empty
    if len(fields) == 0:
      continue

    # Identify start of new reactor data
    if fields[0] == "Time(min)":

      # Save if non-empty
      if data:
        reactors[N_reactor] = np.array(data)
      N_reactor += 1

    # Get the numbers
    try:
      data += [[float(fields[0]),float(fields[1])]]
    except:
      pass

    #
    # DELETED THE DIAGNOSTIC BREAK STATEMENT
    #
print("reactors.keys(): {}".format(reactors.keys()))
print("lengths: {}".format([ len(reactors[i]) for i in reactors.keys() ]))


reactors.keys(): dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8])
lengths: [481, 962, 1443, 1924, 2405, 2886, 3367, 3848, 4329]


There are no formal errors, but we can see two problems with the output. First, we see that the amount of data for each reactor is growing, but if we inspect the file we see that it should be constant. Can you see why this is happening?

The problem is that we aren't resetting the `data` list between reactors. So the previous reactors data is still there when we start parsing the next. We can correct this by reinitializing the `data` list when we save the reactor data to the dictionary:


In [17]:
data = [] # will temporarily hold each reactor's data
N_reactor = -1 # keeps track of which reactor we are parsing
reactors = {} # dictionary for holding each reactor's data as an array
with open("mult_reactors.txt",'r') as f:
  for lines in f:
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Skip empty
    if len(fields) == 0:
      continue

    # Identify start of new reactor data
    if fields[0] == "Time(min)":

      # Save if non-empty
      if data:
        reactors[N_reactor] = np.array(data)
        data = [] # reinitialize data after we save it
      N_reactor += 1

    # Get the numbers
    try:
      data += [[float(fields[0]),float(fields[1])]]
    except:
      pass

print("reactors.keys(): {}".format(reactors.keys()))
print("lengths: {}".format([ len(reactors[i]) for i in reactors.keys() ]))


reactors.keys(): dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8])
lengths: [481, 481, 481, 481, 481, 481, 481, 481, 481]


Reinitializing `data` after saving each reactor corrects our problem. The second problem is with the number of reactors. If we inspect the file we see that there are 10 reactors, but our final dictionary only has data for 9. What's the problem?

The last reactor never gets saved because it isn't followed by a `"Time(min)"` header. This is a common issue with parsers with chained data. The last one won't get saved because the file just ends. We can correct this by saving `data` to the dictionary after the loop breaks:


In [18]:
data = [] # will temporarily hold each reactor's data
N_reactor = -1 # keeps track of which reactor we are parsing
reactors = {} # dictionary for holding each reactor's data as an array
with open("mult_reactors.txt",'r') as f:
  for lines in f:
    fields = lines.split() # the file is space delimitted so this breaks up the lines into words

    # Skip empty
    if len(fields) == 0:
      continue

    # Identify start of new reactor data
    if fields[0] == "Time(min)":

      # Save if non-empty
      if data:
        reactors[N_reactor] = np.array(data)
        data = [] # reinitialize data after we save it
      N_reactor += 1

    # Get the numbers
    try:
      data += [[float(fields[0]),float(fields[1])]]
    except:
      pass

# Save the last reactor
reactors[N_reactor] = data

print("reactors.keys(): {}".format(reactors.keys()))
print("lengths: {}".format([ len(reactors[i]) for i in reactors.keys() ]))
print("reactors[9][-1]: {}".format(reactors[9][-1]))


reactors.keys(): dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
lengths: [481, 481, 481, 481, 481, 481, 481, 481, 481, 481]
reactors[9][-1]: [480.0, 89.613282]


We now have a working solution for our problem. 

Review the final program and think about explaining it to someone just starting on the problem. If you try to explain it in the order it is written, there are several things here that are difficult to explain. 

For example, we initialize `data=[]` and `reactors={}` at the beginning. It isn't obvious that we will need both, the first for temporarily holding each reactor's data, and the second for storing the final data. But during iterative development we converged on this solution. Likewise, it isn't obvious that `if len(fields) == 0:` is the first thing that we would need to check about the line. We added this because we encountered an error while testing a version of the program without it. Another example is the `if data:` statement. We added this because we needed to skip saving the data during the first encounter with the header, but it would be difficult to explain to someone that hadn't gone through the iterative process with us. 

A lot of programming is asynchronous like this. Reading the final solution from start to finish requires explanations about things that happen later. This is why it is VERY DIFFICULT to write a multi-step function or for loop correctly on the first try. We are used to thinking sequentially. To develop your programming intuitition, you need to break problems down and iteratively build up the behavior you need. 


## References
- Python file I/O: https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files
- NumPy text I/O: https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html
- NumPy genfromtxt: https://numpy.org/doc/stable/reference/generated/numpy.genfromtxt.html
- Pandas read_csv: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
